# Component 1 — RQ1–RQ4 Results

**J26-SE-325**, liquidity-aware forecasting and portfolio optimization engine.

This notebook is a **presentation layer only**. Every number is computed by
`experiments/run_rq_analysis.py` and logged to MLflow, so results are reproducible from a
terminal without opening Jupyter, and the dissertation can pull tables without re-running
anything by hand.

| RQ | Question | Needs a trained model? |
|----|----------|------------------------|
| RQ1 | Does the hybrid beat the LSTM baseline and zero-shot foundation models? | **Yes** — Colab fine-tuning |
| RQ2 | Does MOEA/D beat mean-variance at equal or lower cost? | No |
| RQ3 | Does the fuzzy GA reduce slippage vs naive liquidation? | No |
| RQ4 | How does plan quality degrade under stress, and where does it break? | No |

In [ ]:
import sys, warnings, logging
from pathlib import Path

warnings.filterwarnings('ignore')
logging.disable(logging.INFO)
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'experiments' else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda v: f'{v:,.6f}')
plt.rcParams.update({'figure.figsize': (10, 4.5), 'axes.grid': True, 'grid.alpha': 0.3})

## RQ1 — Forecast quality

Reports which forecasters this environment can actually run. The full comparison
(LSTM baseline vs zero-shot vs LoRA-tuned vs hybrid, split by asset class) needs a
fine-tuning run; on a CPU-only machine that belongs in Colab.

Once adapters exist, populate with `evaluation.backtest.run_walk_forward`, which enforces
no look-ahead per fold.

In [ ]:
from forecasting.base import available_foundation_models, registered_forecasters

print('usable forecasters:', registered_forecasters())
print('foundation models :', available_foundation_models())

In [ ]:
# Walk-forward backtest, once a hybrid adapter is trained. Left un-run here because it
# takes hours on CPU; the harness itself is covered by tests/test_evaluation.py.
#
# from evaluation.backtest import BacktestConfig, run_walk_forward
# from evaluation.metrics import forecast_metrics
#
# preds = run_walk_forward(feature_table, 'hybrid', horizon=5,
#                          config=BacktestConfig(embargo_days=5))
# forecast_metrics(preds['target_return'].to_numpy(),
#                  preds[['p10','p50','p90']].to_numpy(), (0.1, 0.5, 0.9))

## RQ2 — Allocation quality: MOEA/D vs Markowitz

The claim is *equal or better return at equal or lower realized cost*. Beating the baseline
by simply trading more would not support it, which is why liquidity cost is reported
alongside return rather than folded into a single score.

In [ ]:
from experiments.run_rq_analysis import rq2_allocation_comparison

rq2 = rq2_allocation_comparison()
rq2[['method', 'selection_rule', 'expected_return', 'liquidity_cost', 'thin_weight']]

In [ ]:
baseline = rq2[rq2.method == 'markowitz_max_sharpe'].iloc[0]
best = rq2[rq2.method == 'moead_liquidity_aware'].nsmallest(1, 'liquidity_cost').iloc[0]

print(f"Markowitz : return {baseline.expected_return:.6f}  cost {baseline.liquidity_cost:.6f}")
print(f"MOEA/D    : return {best.expected_return:.6f}  cost {best.liquidity_cost:.6f}")
print(f"\ncost reduction: {(1 - best.liquidity_cost / baseline.liquidity_cost) * 100:.1f}% "
      f"at {(best.expected_return / baseline.expected_return - 1) * 100:+.1f}% return")

# Note the mechanism: liquidity cost is charged on the TRADE, not the holding. MOEA/D can
# hold MORE of an illiquid name than Markowitz and still pay less, by not rebalancing it.
print(f"\nTHIN weight  Markowitz {baseline.thin_weight:.4f} | MOEA/D {best.thin_weight:.4f}"
      f"  (current holding 0.1000)")

In [ ]:
# Sensitivity: the Pareto selection rule is a judgement call, so report how much it moves
# the answer rather than silently picking one.
moead = rq2[rq2.method == 'moead_liquidity_aware']
ax = moead.plot.bar(x='selection_rule', y=['expected_return', 'liquidity_cost'],
                    secondary_y='liquidity_cost', rot=0,
                    title='RQ2 sensitivity to the Pareto selection rule')
plt.tight_layout(); plt.show()

## RQ3 — Fuzzy GA vs naive liquidation

Swept over 3 withdrawal sizes × 3 urgency levels. A single cell would be cherry-picking:
the advantage is largest exactly where the deadline constraint binds.

In [ ]:
from experiments.run_rq_analysis import rq3_withdrawal_vs_naive

rq3 = rq3_withdrawal_vs_naive()
summary = (rq3.groupby('baseline')[['absolute_improvement', 'relative_improvement_pct']]
              .mean().sort_values('relative_improvement_pct', ascending=False))
summary

In [ ]:
pivot = rq3.pivot_table(index='urgency', columns='baseline',
                        values='relative_improvement_pct', aggfunc='mean')
ax = pivot.plot(marker='o', title='RQ3: fuzzy-GA advantage vs each baseline, by urgency')
ax.set_ylabel('reduction in realized loss (%)'); ax.set_xlabel('withdrawal urgency')
ax.axhline(0, color='k', lw=0.8)
plt.tight_layout(); plt.show()

# most-liquid-first is the honest opponent: it is already liquidity-aware, just myopically.
# pro-rata is the common default and ignores liquidity entirely.

## RQ4 — Degradation under stress

`total_cost = realized_loss + shortfall`. Reporting loss alone would flatter a method that
simply gives up early and therefore incurs no slippage.

In [ ]:
from experiments.run_rq_analysis import rq4_stress_degradation

rq4_results, rq4_summary = rq4_stress_degradation()
rq4_summary

In [ ]:
compound = rq4_results[rq4_results.scenario_type == 'compound']
curve = compound.pivot_table(index='severity', columns='method', values='total_cost')

ax = curve.plot(marker='o', logy=True,
                title='RQ4: degradation under compound stress (log scale)')
ax.set_ylabel('total cost = realized loss + shortfall'); ax.set_xlabel('stress severity')
plt.tight_layout(); plt.show()

### RQ4 — the honest reading

The fuzzy GA has the **best unstressed cost** but degrades faster than most-liquid-first.
Only pro-rata ever becomes infeasible.

Diagnosis, not hand-waving: **the fuzzy layer saturates under severe stress.** The cell
below shows every holding collapsing to an identical `sell_priority`, at which point the
priority signal carries no ordering information.

Two causes — one fixed, one deliberately left open:

1. *Fixed.* `position_liquidity_score` was linear and saturated at 20% of ADV, so a 95% ADV
   collapse floored every holding to 0. Now log-scaled over five decades.
2. *Open.* The rule base maps all of `(HIGH, TURBULENT, {LIQUID, NORMAL, ILLIQUID})` to
   `VERY_HIGH`. Defensible as crisis behaviour, but it removes the discrimination RQ4
   measures. **A methodology decision for supervisor discussion — not tuned until the
   proposed method wins.**

In [ ]:
from experiments.run_rq_analysis import DEMO_PORTFOLIO
from optimization.fuzzy_withdrawal import compute_portfolio_priorities
from optimization.stress_scenarios import ScenarioType, apply_to_holdings, make_scenario

rows = []
for severity in (0.0, 0.33, 0.66, 1.0):
    stressed = apply_to_holdings(DEMO_PORTFOLIO, make_scenario(ScenarioType.COMPOUND, severity))
    priorities = compute_portfolio_priorities(stressed, withdrawal_urgency=0.7)
    row = {s: round(r.sell_priority, 1) for s, r in priorities.items()}
    row['severity'] = severity
    row['spread'] = round(max(row[s] for s in DEMO_PORTFOLIO) - min(row[s] for s in DEMO_PORTFOLIO), 1)
    rows.append(row)

saturation = pd.DataFrame(rows).set_index('severity')
print('sell_priority per holding, and the spread between them:')
saturation

## Reproducibility

Every run is logged to MLflow (`sqlite:///artifacts/mlflow.db`). Inspect with:

```bash
uv run mlflow ui --backend-store-uri sqlite:///artifacts/mlflow.db
```

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///artifacts/mlflow.db')
for experiment in mlflow.search_experiments():
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    if len(runs):
        print(f'{experiment.name}: {len(runs)} run(s)')